# Task 2 - Data Profiling and Cleaning

This notebook profiles and cleans the raw KAUST and KFUPM research datasets.

The raw source files are preserved unchanged in `data/raw/`.
Cleaned outputs are written to `data/interim/`.

In [1]:
import pandas as pd
import re
import html
import json
from pathlib import Path

In [2]:
raw_dir = Path("../data/raw")
interim_dir = Path("../data/interim")

interim_dir.mkdir(parents=True, exist_ok=True)

In [3]:
def safe_unique_count(series):
    values = series.dropna().apply(
        lambda x: json.dumps(x, sort_keys=True)
        if isinstance(x, (list, dict))
        else str(x)
    )
    return values.nunique()


## KAUST 2023 - Data Profiling

The raw KAUST 2023 dataset is loaded from `data/raw/` for data discovery and profiling before any cleaning is applied.

In [4]:
kaust_2023 = pd.read_csv(
    raw_dir / "KAUST_2023_raw.csv",
    low_memory=False
)

kaust_2023.head()

,Type,Title,Authors,Journal,Publisher,DOI,Handle,Publication Date,Citation,Abstract,Link to License,Status,Link to PDF,Link to Extracted Text,Metadata Last Modified
0,Preprint,Adsorption of Brilliant Blue Dye on Biochar fr...,"Areej, Azwa; Mumtaz, Maimoona; Aman, Fariha; B...",NaN,Research Square Platform LLC,10.21203/rs.3.rs-2796188/v1,http://hdl.handle.net/10754/691095,2023-04-09,"Areej, A., Mumtaz, M., Aman, F., BADSHAH, S., ...",Here we describe the utilization of freshwater...,https://creativecommons.org/licenses/by/4.0/,Under Review,https://repository.kaust.edu.sa/bitstream/1075...,https://repository.kaust.edu.sa/bitstream/hand...,2023-04-13T17:00:09
1,Preprint,VARS: Video Assistant Referee System for Autom...,"Held, Jan; Cioppa, Anthony; Giancola, Silvio; ...",NaN,arXiv,NaN,http://hdl.handle.net/10754/691094,2023-04-10,NaN,The Video Assistant Referee (VAR) has revoluti...,NaN,Under Review,https://repository.kaust.edu.sa/bitstream/1075...,https://repository.kaust.edu.sa/bitstream/hand...,2023-04-13T16:30:12
2,Preprint,SoccerNet-Caption: Dense Video Captioning for ...,"Mkhallati, Hassan; Cioppa, Anthony; Giancola, ...",NaN,arXiv,NaN,http://hdl.handle.net/10754/691093,2023-04-10,NaN,Soccer is more than just a game - it is a pass...,NaN,Under Review,https://repository.kaust.edu.sa/bitstream/1075...,https://repository.kaust.edu.sa/bitstream/hand...,2023-04-13T16:30:12
3,Preprint,Towards Active Learning for Action Spotting in...,"Giancola, Silvio; Cioppa, Anthony; Georgieva, ...",NaN,arXiv,NaN,http://hdl.handle.net/10754/691091,2023-04-09,NaN,Association football is a complex and dynamic ...,NaN,Under Review,https://repository.kaust.edu.sa/bitstream/1075...,https://repository.kaust.edu.sa/bitstream/hand...,2023-04-13T16:10:11
4,Preprint,Time and Energy Constrained Large-Scale IoT Ne...,"Emara, Mostafa Lotfy; Kouzayha, Nour Hicham; E...",NaN,arXiv,NaN,http://hdl.handle.net/10754/691090,2023-04-09,NaN,Closed-loop rate adaptation and error-control ...,NaN,Under Review,https://repository.kaust.edu.sa/bitstream/1075...,https://repository.kaust.edu.sa/bitstream/hand...,2023-04-13T16:00:08


In [5]:
print("Rows:", kaust_2023.shape[0])
print("Columns:", kaust_2023.shape[1])

Rows: 31086
Columns: 15


In [6]:
kaust_2023.columns.tolist()

['Type',
 'Title',
 'Authors',
 'Journal',
 'Publisher',
 'DOI',
 'Handle',
 'Publication Date',
 'Citation',
 'Abstract',
 'Link to License',
 'Status',
 'Link to PDF',
 'Link to Extracted Text',
 'Metadata Last Modified']

In [7]:
kaust_2023.dtypes

Type                      str
Title                     str
Authors                   str
Journal                   str
Publisher                 str
DOI                       str
Handle                    str
Publication Date          str
Citation                  str
Abstract                  str
Link to License           str
Status                    str
Link to PDF               str
Link to Extracted Text    str
Metadata Last Modified    str
dtype: object

In [8]:
kaust_2023.isnull().sum()

Type                         16
Title                         0
Authors                      37
Journal                    7116
Publisher                  1683
DOI                        3482
Handle                        1
Publication Date             41
Citation                   3519
Abstract                   1608
Link to License           25689
Status                        0
Link to PDF               14698
Link to Extracted Text    14698
Metadata Last Modified        0
dtype: int64

In [9]:
print("Duplicate rows:", kaust_2023.duplicated().sum())

Duplicate rows: 0


In [10]:
profile_summary = pd.DataFrame({
    "column": kaust_2023.columns,
    "dtype": kaust_2023.dtypes.astype(str).values,
    "null_count": kaust_2023.isnull().sum().values,
    "null_percent": (
        kaust_2023.isnull().mean().values * 100
    ).round(2),
    "unique_count": kaust_2023.nunique(dropna=True).values,
    "sample_value": [
        kaust_2023[col].dropna().iloc[0]
        if not kaust_2023[col].dropna().empty
        else None
        for col in kaust_2023.columns
    ]
})

profile_summary

,column,dtype,null_count,null_percent,unique_count,sample_value
0,Type,str,16,0.05,30,Preprint
1,Title,str,0,0.00,30679,Adsorption of Brilliant Blue Dye on Biochar fr...
2,Authors,str,37,0.12,25657,"Areej, Azwa; Mumtaz, Maimoona; Aman, Fariha; B..."
3,Journal,str,7116,22.89,4687,ACS Applied Energy Materials
4,Publisher,str,1683,5.41,658,Research Square Platform LLC
5,DOI,str,3482,11.20,27556,10.21203/rs.3.rs-2796188/v1
6,Handle,str,1,0.00,31083,http://hdl.handle.net/10754/691095
7,Publication Date,str,41,0.13,4347,2023-04-09
8,Citation,str,3519,11.32,27532,"Areej, A., Mumtaz, M., Aman, F., BADSHAH, S., ..."
9,Abstract,str,1608,5.17,28560,Here we describe the utilization of freshwater...


### Data Quality Issues and Decisions

1. Some records have missing authors.
   - Decision: Keep the records and allow authors to remain nullable.

2. Journal information is missing for some records.
   - Decision: Keep the records because some publication types may not have a journal.

3. DOI is missing for some records.
   - Decision: Keep DOI nullable because not every research record has a DOI.

4. A small number of publication dates are missing.
   - Decision: Convert valid dates to datetime and keep invalid or missing values as null.

5. License information is missing for many records.
   - Decision: Keep the field nullable and document the limitation.

6. PDF and extracted-text links are missing for some records.
   - Decision: Keep these fields nullable because they are optional metadata.

7. Column names are not standardized.
   - Decision: Convert all column names to snake_case.

8. Date columns are currently stored as strings.
   - Decision: Convert publication_date and metadata_last_modified to datetime.

9. Potential duplicate publications need to be investigated.
   - Decision: Check duplicates using title, DOI, and repository handle before removing any records.

In [11]:
kaust_2023_clean = kaust_2023.copy()

In [12]:
print("Raw rows:", len(kaust_2023))
print("Cleaning copy rows:", len(kaust_2023_clean))

Raw rows: 31086
Cleaning copy rows: 31086


In [13]:
kaust_2023_clean.columns = (
    kaust_2023_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

kaust_2023_clean.columns.tolist()

['type',
 'title',
 'authors',
 'journal',
 'publisher',
 'doi',
 'handle',
 'publication_date',
 'citation',
 'abstract',
 'link_to_license',
 'status',
 'link_to_pdf',
 'link_to_extracted_text',
 'metadata_last_modified']

In [14]:
text_columns = kaust_2023_clean.select_dtypes(include="object").columns

for column in text_columns:
    kaust_2023_clean[column] = kaust_2023_clean[column].str.strip()

C:\Users\nawaf\AppData\Local\Temp\ipykernel_18408\1133887846.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = kaust_2023_clean.select_dtypes(include="object").columns


In [15]:
kaust_2023_clean = kaust_2023_clean.replace(r"^\s*$", pd.NA, regex=True)

In [16]:
kaust_2023_clean["publication_date"] = pd.to_datetime(
    kaust_2023_clean["publication_date"],
    format="mixed",
    errors="coerce",
    utc=True
)

kaust_2023_clean["metadata_last_modified"] = pd.to_datetime(
    kaust_2023_clean["metadata_last_modified"],
    format="mixed",
    errors="coerce",
    utc=True
)

In [17]:
kaust_2023_clean["publication_year"] = (
    kaust_2023_clean["publication_date"].dt.year.astype("Int64")
)

kaust_2023_clean = kaust_2023_clean[
    kaust_2023_clean["publication_year"] == 2023
].copy()

print("KAUST 2023 rows:", len(kaust_2023_clean))

KAUST 2023 rows: 929


In [18]:
kaust_2023_clean[
    ["publication_date", "metadata_last_modified"]
].dtypes

publication_date          datetime64[us, UTC]
metadata_last_modified    datetime64[us, UTC]
dtype: object

In [19]:
print(
    "Missing publication dates after conversion:",
    kaust_2023_clean["publication_date"].isna().sum()
)

print(
    "Missing metadata dates after conversion:",
    kaust_2023_clean["metadata_last_modified"].isna().sum()
)

Missing publication dates after conversion: 0
Missing metadata dates after conversion: 0


In [20]:
print(
    "Duplicate titles:",
    kaust_2023_clean.duplicated(subset=["title"]).sum()
)

print(
    "Duplicate DOI:",
    kaust_2023_clean[
        kaust_2023_clean["doi"].notna()
    ].duplicated(subset=["doi"]).sum()
)

print(
    "Duplicate handles:",
    kaust_2023_clean[
        kaust_2023_clean["handle"].notna()
    ].duplicated(subset=["handle"]).sum()
)

Duplicate titles: 3
Duplicate DOI: 1
Duplicate handles: 0


In [21]:
duplicate_handles = kaust_2023_clean[
    kaust_2023_clean["handle"].notna()
    & kaust_2023_clean.duplicated(
        subset=["handle"],
        keep=False
    )
].sort_values("handle")

duplicate_handles[
    [
        "title",
        "doi",
        "handle",
        "type",
        "publication_date"
    ]
]

,title,doi,handle,type,publication_date


In [22]:
duplicate_dois = kaust_2023_clean[
    kaust_2023_clean["doi"].notna()
    & kaust_2023_clean.duplicated(
        subset=["doi"],
        keep=False
    )
].sort_values("doi")

duplicate_dois[
    [
        "title",
        "doi",
        "handle",
        "type",
        "publication_date"
    ]
].head(20)

,title,doi,handle,type,publication_date
328,Down-converting luminescent optoelectronics an...,10.1063/5.0127552,http://hdl.handle.net/10754/690319,Article,2023-02-22 00:00:00+00:00
718,Down-converting luminescent optoelectronics an...,10.1063/5.0127552,http://hdl.handle.net/10754/687547,Article,2023-02-22 00:00:00+00:00


In [23]:
duplicate_doi_check = duplicate_dois[
    [
        "title",
        "doi",
        "handle",
        "type",
        "publication_date",
        "metadata_last_modified"
    ]
].sort_values(["doi", "publication_date"])

duplicate_doi_check.head(30)

,title,doi,handle,type,publication_date,metadata_last_modified
328,Down-converting luminescent optoelectronics an...,10.1063/5.0127552,http://hdl.handle.net/10754/690319,Article,2023-02-22 00:00:00+00:00,2023-03-14 09:20:09+00:00
718,Down-converting luminescent optoelectronics an...,10.1063/5.0127552,http://hdl.handle.net/10754/687547,Article,2023-02-22 00:00:00+00:00,2023-03-14 09:50:12+00:00


In [24]:
exact_duplicate_candidates = kaust_2023_clean[
    kaust_2023_clean["doi"].notna()
].duplicated(
    subset=[
        "doi",
        "title",
        "type",
        "publication_date"
    ],
    keep=False
)

print(
    "Exact duplicate candidates:",
    exact_duplicate_candidates.sum()
)

Exact duplicate candidates: 2


In [25]:
duplicate_groups = (
    kaust_2023_clean[
        kaust_2023_clean["doi"].notna()
    ]
    .groupby(
        ["doi", "title", "type", "publication_date"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
)

duplicate_groups = duplicate_groups[
    duplicate_groups["record_count"] > 1
]

print("Duplicate groups:", len(duplicate_groups))
print(
    "Potential rows to remove:",
    (duplicate_groups["record_count"] - 1).sum()
)

Duplicate groups: 1
Potential rows to remove: 1


In [26]:
duplicate_records = kaust_2023_clean.merge(
    duplicate_groups[
        ["doi", "title", "type", "publication_date"]
    ],
    on=["doi", "title", "type", "publication_date"],
    how="inner"
)

duplicate_records[
    [
        "title",
        "doi",
        "type",
        "publication_date",
        "handle",
        "metadata_last_modified"
    ]
]

,title,doi,type,publication_date,handle,metadata_last_modified
0,Down-converting luminescent optoelectronics an...,10.1063/5.0127552,Article,2023-02-22 00:00:00+00:00,http://hdl.handle.net/10754/690319,2023-03-14 09:20:09+00:00
1,Down-converting luminescent optoelectronics an...,10.1063/5.0127552,Article,2023-02-22 00:00:00+00:00,http://hdl.handle.net/10754/687547,2023-03-14 09:50:12+00:00


In [27]:
with_doi = kaust_2023_clean[
    kaust_2023_clean["doi"].notna()
].copy()

without_doi = kaust_2023_clean[
    kaust_2023_clean["doi"].isna()
].copy()

with_doi["completeness_score"] = (
    with_doi.notna().sum(axis=1)
)

with_doi = (
    with_doi
    .sort_values(
        ["completeness_score", "metadata_last_modified"],
        ascending=[False, False]
    )
    .drop_duplicates(
        subset=["doi", "title", "type", "publication_date"],
        keep="first"
    )
    .drop(columns="completeness_score")
)

kaust_2023_clean = pd.concat(
    [with_doi, without_doi],
    ignore_index=True
)

print("Rows after dedup:", len(kaust_2023_clean))

Rows after dedup: 928


In [28]:
print(
    "Remaining duplicates:",
    kaust_2023_clean[
        kaust_2023_clean["doi"].notna()
    ].duplicated(
        subset=["doi", "title", "type", "publication_date"]
    ).sum()
)

Remaining duplicates: 0


In [29]:
print("Missing DOI:", kaust_2023_clean["doi"].isna().sum())
print("Missing Handle:", kaust_2023_clean["handle"].isna().sum())

Missing DOI: 113
Missing Handle: 0


In [30]:
print(
    "Duplicate Handles:",
    kaust_2023_clean["handle"].duplicated().sum()
)

Duplicate Handles: 0


In [31]:
kaust_2023_clean["research_id"] = kaust_2023_clean["handle"]

kaust_2023_clean["university"] = "KAUST"
kaust_2023_clean["research_field"] = pd.NA
kaust_2023_clean["tech_category"] = pd.NA
kaust_2023_clean["url"] = kaust_2023_clean["handle"]
kaust_2023_clean["source"] = "KAUST Repository"

In [32]:
print("Duplicate research IDs:", kaust_2023_clean["research_id"].duplicated().sum())
print("Missing research IDs:", kaust_2023_clean["research_id"].isna().sum())

Duplicate research IDs: 0
Missing research IDs: 0


### Standardize Publication Date Format

The repository dates include a time and UTC offset (e.g. `2023-04-11 00:00:00+00:00`). Only the calendar date is kept, in `YYYY-MM-DD` format, to match the team schema.

In [33]:
kaust_2023_clean["publication_date"] = (
    kaust_2023_clean["publication_date"].dt.strftime("%Y-%m-%d")
)

populated = kaust_2023_clean["publication_date"].dropna()
assert populated.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()

print("Dates in YYYY-MM-DD format:", len(populated))
print("Missing publication dates:", kaust_2023_clean["publication_date"].isna().sum())
print("Sample:", populated.head(3).tolist())

Dates in YYYY-MM-DD format: 928
Missing publication dates: 0
Sample: ['2023-04-11', '2023-04-11', '2023-03-10']


In [34]:
kaust_2023_clean = kaust_2023_clean[
    [
        "research_id",
        "university",
        "title",
        "authors",
        "publication_year",
        "publication_date",
        "abstract",
        "research_field",
        "tech_category",
        "journal",
        "doi",
        "url",
        "source"
    ]
].copy()

In [35]:
print("Final rows:", len(kaust_2023_clean))
print("Final columns:", len(kaust_2023_clean.columns))

kaust_2023_clean.head()

Final rows: 928
Final columns: 13


,research_id,university,title,authors,publication_year,publication_date,abstract,research_field,tech_category,journal,doi,url,source
0,http://hdl.handle.net/10754/691073,KAUST,Elucidating the Role of Contact-Induced Gap St...,"Pradhan, Rakesh R.; Eswaran, Mathan Kumar; Sub...",2023,2023-04-11,Metal halide perovskite solar cells hold great...,<NA>,<NA>,ACS Applied Energy Materials,10.1021/acsaem.3c00292,http://hdl.handle.net/10754/691073,KAUST Repository
1,http://hdl.handle.net/10754/691069,KAUST,Enhanced Organic Electrochemical Transistor Pe...,"Ding, Bowen; Jo, Il-Young; Yu, Hang; Kim, Ji H...",2023,2023-04-11,Emergent bioelectronic technologies are underp...,<NA>,<NA>,Chemistry of Materials,10.1021/acs.chemmater.3c00327,http://hdl.handle.net/10754/691069,KAUST Repository
2,http://hdl.handle.net/10754/691064,KAUST,Semi-universal geo-crack detection by machine ...,"Shi, Yongxiang; Ballesio, Marco; Johansen, Kas...",2023,2023-03-10,Introduction: Cracks are a key feature that de...,<NA>,<NA>,Frontiers in Earth Science,10.3389/feart.2023.1073211,http://hdl.handle.net/10754/691064,KAUST Repository
3,http://hdl.handle.net/10754/691054,KAUST,Structural Changes in Nonlocal Denoising Model...,"Davoli, Elisa; Ferreira, Rita; Kreisbeck, Caro...",2023,2023-04-10,We introduce a unified framework based on bi-l...,<NA>,<NA>,Applied Mathematics & Optimization,10.1007/s00245-023-09982-4,http://hdl.handle.net/10754/691054,KAUST Repository
4,http://hdl.handle.net/10754/691052,KAUST,Advances in One-Pot Chiral Amine Synthesis Ena...,"Mathew, Sam; Renn, Dominik; Rueping, Magnus",2023,2023-04-10,Amine transaminases constitute an important cl...,<NA>,<NA>,ACS Catalysis,10.1021/acscatal.3c00555,http://hdl.handle.net/10754/691052,KAUST Repository


In [36]:
print("Rows:", len(kaust_2023_clean))
print(
    "Duplicate research IDs:",
    kaust_2023_clean["research_id"].duplicated().sum()
)
print(
    "Missing research IDs:",
    kaust_2023_clean["research_id"].isna().sum()
)

kaust_2023_clean.head()

Rows: 928
Duplicate research IDs: 0
Missing research IDs: 0


,research_id,university,title,authors,publication_year,publication_date,abstract,research_field,tech_category,journal,doi,url,source
0,http://hdl.handle.net/10754/691073,KAUST,Elucidating the Role of Contact-Induced Gap St...,"Pradhan, Rakesh R.; Eswaran, Mathan Kumar; Sub...",2023,2023-04-11,Metal halide perovskite solar cells hold great...,<NA>,<NA>,ACS Applied Energy Materials,10.1021/acsaem.3c00292,http://hdl.handle.net/10754/691073,KAUST Repository
1,http://hdl.handle.net/10754/691069,KAUST,Enhanced Organic Electrochemical Transistor Pe...,"Ding, Bowen; Jo, Il-Young; Yu, Hang; Kim, Ji H...",2023,2023-04-11,Emergent bioelectronic technologies are underp...,<NA>,<NA>,Chemistry of Materials,10.1021/acs.chemmater.3c00327,http://hdl.handle.net/10754/691069,KAUST Repository
2,http://hdl.handle.net/10754/691064,KAUST,Semi-universal geo-crack detection by machine ...,"Shi, Yongxiang; Ballesio, Marco; Johansen, Kas...",2023,2023-03-10,Introduction: Cracks are a key feature that de...,<NA>,<NA>,Frontiers in Earth Science,10.3389/feart.2023.1073211,http://hdl.handle.net/10754/691064,KAUST Repository
3,http://hdl.handle.net/10754/691054,KAUST,Structural Changes in Nonlocal Denoising Model...,"Davoli, Elisa; Ferreira, Rita; Kreisbeck, Caro...",2023,2023-04-10,We introduce a unified framework based on bi-l...,<NA>,<NA>,Applied Mathematics & Optimization,10.1007/s00245-023-09982-4,http://hdl.handle.net/10754/691054,KAUST Repository
4,http://hdl.handle.net/10754/691052,KAUST,Advances in One-Pot Chiral Amine Synthesis Ena...,"Mathew, Sam; Renn, Dominik; Rueping, Magnus",2023,2023-04-10,Amine transaminases constitute an important cl...,<NA>,<NA>,ACS Catalysis,10.1021/acscatal.3c00555,http://hdl.handle.net/10754/691052,KAUST Repository


In [37]:
kaust_2023_clean.isna().sum()

research_id           0
university            0
title                 0
authors               1
publication_year      0
publication_date      0
abstract             20
research_field      928
tech_category       928
journal             285
doi                 113
url                   0
source                0
dtype: int64

In [38]:
print("Missing titles:", kaust_2023_clean["title"].isna().sum())
print("Missing authors:", kaust_2023_clean["authors"].isna().sum())
print("Missing publication dates:", kaust_2023_clean["publication_date"].isna().sum())
print("Missing research IDs:", kaust_2023_clean["research_id"].isna().sum())

Missing titles: 0
Missing authors: 1
Missing publication dates: 0
Missing research IDs: 0


In [39]:
output_file = interim_dir / "KAUST_2023_cleaned.csv"

kaust_2023_clean.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: ..\data\interim\KAUST_2023_cleaned.csv


In [40]:
saved_kaust_2023 = pd.read_csv(output_file)

print("Saved rows:", len(saved_kaust_2023))
print("Saved columns:", len(saved_kaust_2023.columns))

Saved rows: 928
Saved columns: 13


## KAUST 2024–2025 - Crossref Profiling and Cleaning

The raw Crossref JSON files are loaded and combined for profiling before flattening and cleaning.

In [41]:
crossref_files = sorted(
    raw_dir.glob("KAUST_Crossref_2024_2025_page_*.json")
)

len(crossref_files)

2

In [42]:
crossref_records = []

for file in crossref_files:
    with open(file, "r", encoding="utf-8") as f:
        page_data = json.load(f)

    crossref_records.extend(
        page_data["message"]["items"]
    )

print("Total records:", len(crossref_records))

Total records: 124


In [43]:
kaust_crossref = pd.json_normalize(crossref_records)

print("Rows:", kaust_crossref.shape[0])
print("Columns:", kaust_crossref.shape[1])

Rows: 124
Columns: 78


In [44]:
kaust_crossref.head()

,reference-count,publisher,issue,license,funder,short-container-title,abstract,DOI,type,page,...,review.type,review.stage,relation.is-review-of,relation.is-financed-by,edition-number,isbn-type,ISBN,editor,updated-by,contributor
0,16,Optica Publishing Group,5,"[{'start': {'date-parts': [[2024, 2, 20]], 'da...","[{'DOI': '10.13039/501100001665', 'name': 'Age...",[Opt. Express],<jats:p>We utilized a metal propionate solutio...,10.1364/oe.503864,journal-article,7651,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,38,Optica Publishing Group,3,"[{'start': {'date-parts': [[2024, 1, 23]], 'da...",NaN,[Opt. Express],<jats:p>\n In this manuscri...,10.1364/oe.511412,journal-article,4102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,32,International Union of Crystallography (IUCr),2,"[{'start': {'date-parts': [[2024, 2, 9]], 'dat...","[{'DOI': '10.13039/501100001665', 'name': 'Age...","[J Synchrotron Rad, J Synchrotron Radiat]",<jats:p>\n X-ray ptychograp...,10.1107/s160057752400016x,journal-article,399-408,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,American Chemical Society (ACS),NaN,"[{'start': {'date-parts': [[2024, 1, 24]], 'da...",NaN,NaN,<jats:p>The increasing use of machine learning...,10.26434/chemrxiv-2024-q9tc4,posted-content,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,Cassyni,NaN,NaN,"[{'DOI': '10.13039/501100004052', 'name': 'Kin...",NaN,<jats:p>Volumetric measurements of three-dimen...,10.52843/cassyni.9wnw31,posted-content,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
kaust_crossref.columns.tolist()

['reference-count',
 'publisher',
 'issue',
 'license',
 'funder',
 'short-container-title',
 'abstract',
 'DOI',
 'type',
 'page',
 'update-policy',
 'source',
 'is-referenced-by-count',
 'title',
 'prefix',
 'volume',
 'author',
 'member',
 'reference',
 'container-title',
 'language',
 'link',
 'score',
 'references-count',
 'URL',
 'ISSN',
 'issn-type',
 'assertion',
 'indexed.date-parts',
 'indexed.date-time',
 'indexed.timestamp',
 'indexed.version',
 'content-domain.domain',
 'content-domain.crossmark-restriction',
 'published-print.date-parts',
 'created.date-parts',
 'created.date-time',
 'created.timestamp',
 'published-online.date-parts',
 'deposited.date-parts',
 'deposited.date-time',
 'deposited.timestamp',
 'resource.primary.URL',
 'issued.date-parts',
 'journal-issue.issue',
 'journal-issue.published-online.date-parts',
 'journal-issue.published-print.date-parts',
 'published.date-parts',
 'alternative-id',
 'accepted.date-parts',
 'subtype',
 'posted.date-parts',
 'gro

In [46]:
crossref_profile = pd.DataFrame({
    "column": kaust_crossref.columns,
    "dtype": kaust_crossref.dtypes.astype(str).values,
    "null_count": kaust_crossref.isnull().sum().values,
    "null_percent": (
        kaust_crossref.isnull().mean().values * 100
    ).round(2),
    "unique_count": [
        safe_unique_count(kaust_crossref[col])
        for col in kaust_crossref.columns
    ],
    "sample_value": [
        str(kaust_crossref[col].dropna().iloc[0])[:100]
        if not kaust_crossref[col].dropna().empty
        else None
        for col in kaust_crossref.columns
    ]
})

crossref_profile


,column,dtype,null_count,null_percent,unique_count,sample_value
0,reference-count,int64,0,0.00,62,16
1,publisher,str,0,0.00,17,Optica Publishing Group
2,issue,str,57,45.97,20,5
3,license,object,14,11.29,103,"[{'start': {'date-parts': [[2024, 2, 20]], 'da..."
4,funder,object,44,35.48,67,"[{'DOI': '10.13039/501100001665', 'name': 'Age..."
...,...,...,...,...,...,...
73,isbn-type,object,123,99.19,1,"[{'value': '9783985470945', 'type': 'print'}, ..."
74,ISBN,object,123,99.19,1,"['9783985470945', '9783985475940']"
75,editor,object,121,97.58,3,"[{'given': 'Marvin', 'family': 'Whiteley', 'se..."
76,updated-by,object,123,99.19,1,"[{'DOI': '10.1128/mbio.03870-25', 'type': 'cor..."


In [47]:
important_columns = [
    "DOI",
    "title",
    "author",
    "type",
    "publisher",
    "container-title",
    "URL",
    "abstract"
]

for column in important_columns:
    if column in kaust_crossref.columns:
        print(
            column,
            "- missing:",
            kaust_crossref[column].isna().sum()
        )

DOI - missing: 0
title - missing: 0
author - missing: 0
type - missing: 0
publisher - missing: 0
container-title - missing: 46
URL - missing: 0
abstract - missing: 16


In [48]:
for column in kaust_crossref.columns:
    sample = kaust_crossref[column].dropna()

    if not sample.empty:
        value = sample.iloc[0]

        if isinstance(value, (list, dict)):
            print(column, "->", type(value).__name__)

license -> list
funder -> list
short-container-title -> list
title -> list
author -> list
reference -> list
container-title -> list
link -> list
ISSN -> list
issn-type -> list
assertion -> list
indexed.date-parts -> list
content-domain.domain -> list
published-print.date-parts -> list
created.date-parts -> list
published-online.date-parts -> list
deposited.date-parts -> list
issued.date-parts -> list
journal-issue.published-online.date-parts -> list
journal-issue.published-print.date-parts -> list
published.date-parts -> list
alternative-id -> list
accepted.date-parts -> list
posted.date-parts -> list
relation.has-preprint -> list
relation.has-review -> list
relation.is-part-of -> list
institution -> list
relation.is-preprint-of -> list
relation.has-version -> list
chair -> list
relation.is-based-on -> list
relation.is-supplemented-by -> list
archive -> list
relation.is-version-of -> list
relation.is-same-as -> list
relation.is-review-of -> list
relation.is-financed-by -> list
isbn-typ

In [49]:
kaust_crossref_clean = kaust_crossref.copy()

In [50]:
def parse_crossref_date(date_parts):
    if not isinstance(date_parts, list) or not date_parts:
        return pd.Series({
            "publication_year": pd.NA,
            "publication_date": pd.NaT
        })

    parts = date_parts[0]

    year = parts[0] if len(parts) >= 1 else pd.NA

    if len(parts) >= 3:
        full_date = pd.Timestamp(
            year=parts[0],
            month=parts[1],
            day=parts[2]
        )
    else:
        full_date = pd.NaT

    return pd.Series({
        "publication_year": year,
        "publication_date": full_date
    })

In [51]:
parsed_dates = kaust_crossref["published.date-parts"].apply(
    parse_crossref_date
)

kaust_crossref_clean["publication_year"] = (
    parsed_dates["publication_year"].astype("Int64")
)

kaust_crossref_clean["publication_date"] = (
    parsed_dates["publication_date"]
)

In [52]:
print(
    "Missing publication years:",
    kaust_crossref_clean["publication_year"].isna().sum()
)

print(
    "Missing full publication dates:",
    kaust_crossref_clean["publication_date"].isna().sum()
)

Missing publication years: 0
Missing full publication dates: 1


In [53]:
missing_date_mask = kaust_crossref_clean["publication_date"].isna()

kaust_crossref.loc[
    missing_date_mask,
    ["DOI", "title", "published.date-parts"]
]

,DOI,title,published.date-parts
115,10.1017/dry.2025.10007,[Biological soil crusts in the Arabian Peninsu...,[[2025]]


In [54]:
def extract_authors(authors):
    if not isinstance(authors, list):
        return pd.NA

    names = []

    for author in authors:
        given = author.get("given", "")
        family = author.get("family", "")
        name = f"{given} {family}".strip()

        if name:
            names.append(name)

    return "; ".join(names)


In [55]:
kaust_crossref_clean["title"] = kaust_crossref["title"].apply(
    lambda x: x[0] if isinstance(x, list) and x else pd.NA
)

kaust_crossref_clean["authors"] = kaust_crossref["author"].apply(
    extract_authors
)

kaust_crossref_clean["journal"] = kaust_crossref["container-title"].apply(
    lambda x: x[0] if isinstance(x, list) and x else pd.NA
)

In [56]:
kaust_crossref_clean[
    ["title", "authors", "journal"]
].head()

,title,authors,journal
0,Functionalization of micro-size garnet at the ...,Issatay Nadinov; Oleksandr Kovalenko; Jean-Luc...,Optics Express
1,Theoretical analysis of graded-index topologic...,Amit Kumar Goyal; Diptimayee Dash; Jasmine Sai...,Optics Express
2,<i>ProSPyX</i>\n : software...,Redhouane Boudjehem; Anico Kulow; Javier Pérez...,Journal of Synchrotron Radiation
3,A reagent-driven visual method for analyzing c...,Mikhail Andronov; Natalia Andronova; Michael W...,NaN
4,Scalar Velocimetry using Cross-Scanning LIF,Sigurdur Thoroddsen,NaN


In [57]:
def clean_markup(value):
    if pd.isna(value):
        return pd.NA

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\\n", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [58]:
kaust_crossref_clean["title"] = (
    kaust_crossref_clean["title"].apply(clean_markup)
)

kaust_crossref_clean["abstract"] = (
    kaust_crossref["abstract"].apply(clean_markup)
)

In [59]:
kaust_crossref_clean[
    ["title", "abstract"]
].head()

,title,abstract
0,Functionalization of micro-size garnet at the ...,We utilized a metal propionate solution to pre...
1,Theoretical analysis of graded-index topologic...,"In this manuscript, what we believe to be a no..."
2,ProSPyX : software for post-processing images ...,X-ray ptychography is a coherent diffraction i...
3,A reagent-driven visual method for analyzing c...,The increasing use of machine learning and art...
4,Scalar Velocimetry using Cross-Scanning LIF,Volumetric measurements of three-dimensional v...


In [60]:
kaust_crossref_clean["doi"] = (
    kaust_crossref["DOI"]
    .astype("string")
    .str.strip()
    .str.lower()
)

kaust_crossref_clean["url"] = (
    kaust_crossref["URL"]
    .astype("string")
    .str.strip()
)

kaust_crossref_clean["research_id"] = kaust_crossref_clean["doi"]

kaust_crossref_clean["university"] = "KAUST"
kaust_crossref_clean["research_field"] = pd.NA
kaust_crossref_clean["tech_category"] = pd.NA
kaust_crossref_clean["source"] = "Crossref"

In [61]:
print("Missing research IDs:", kaust_crossref_clean["research_id"].isna().sum())
print("Duplicate research IDs:", kaust_crossref_clean["research_id"].duplicated().sum())
print("Missing URLs:", kaust_crossref_clean["url"].isna().sum())

Missing research IDs: 0
Duplicate research IDs: 0
Missing URLs: 0


In [62]:
print("Total rows:", len(kaust_crossref_clean))

print(
    kaust_crossref_clean["publication_year"]
    .value_counts(dropna=False)
    .sort_index()
)

Total rows: 124
publication_year
2024    51
2025    73
Name: count, dtype: Int64


### Standardize Publication Date Format

Crossref dates are stored as `YYYY-MM-DD` text to match the team schema. Records with an incomplete date keep an empty value; no month or day is inferred.

In [63]:
kaust_crossref_clean["publication_date"] = (
    pd.to_datetime(kaust_crossref_clean["publication_date"], errors="coerce")
    .dt.strftime("%Y-%m-%d")
)

populated = kaust_crossref_clean["publication_date"].dropna()
assert populated.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()

print("Dates in YYYY-MM-DD format:", len(populated))
print("Missing publication dates:", kaust_crossref_clean["publication_date"].isna().sum())
print("Sample:", populated.head(3).tolist())

Dates in YYYY-MM-DD format: 123
Missing publication dates: 1
Sample: ['2024-02-20', '2024-01-23', '2024-02-09']


In [64]:
kaust_crossref_clean = kaust_crossref_clean[
    [
        "research_id",
        "university",
        "title",
        "authors",
        "publication_year",
        "publication_date",
        "abstract",
        "research_field",
        "tech_category",
        "journal",
        "doi",
        "url",
        "source"
    ]
].copy()

In [65]:
print(
    "Duplicate DOI:",
    kaust_crossref_clean.duplicated(
        subset=["doi"]
    ).sum()
)

print(
    "Duplicate titles:",
    kaust_crossref_clean.duplicated(
        subset=["title"]
    ).sum()
)

Duplicate DOI: 0
Duplicate titles: 10


In [66]:
duplicate_titles = kaust_crossref_clean[
    kaust_crossref_clean.duplicated(
        subset=["title"],
        keep=False
    )
].sort_values("title")

duplicate_titles[
    [
        "title",
        "doi",
        "publication_year",
        "publication_date",
        "journal"
    ]
]

,title,doi,publication_year,publication_date,journal
44,Aberration correction in long GRIN lens-based ...,10.7554/elife.101420.1,2024,2024-10-21,NaN
45,Aberration correction in long GRIN lens-based ...,10.7554/elife.101420,2025,2025-05-02,eLife
63,Aberration correction in long GRIN lens-based ...,10.7554/elife.101420.2,2025,2025-03-14,NaN
67,Aberration correction in long GRIN lens-based ...,10.7554/elife.101420.3,2025,2025-04-09,NaN
74,Aberration correction in long GRIN lens-based ...,10.7554/elife.101420.4,2025,2025-05-02,eLife
65,MMP21 behaves as a fluid flow transported morp...,10.7554/elife.104430.1,2025,2025-04-02,NaN
66,MMP21 behaves as a fluid flow transported morp...,10.7554/elife.104430,2025,2025-04-02,NaN
25,Natural variation in salt-induced changes in r...,10.7554/elife.98896.1,2024,2024-07-25,NaN
26,Natural variation in salt-induced changes in r...,10.7554/elife.98896,2025,2025-03-28,eLife
48,Natural variation in salt-induced changes in r...,10.7554/elife.98896.2,2024,2024-11-27,NaN


In [67]:
print(
    "Fully duplicated rows:",
    kaust_crossref_clean.duplicated().sum()
)

Fully duplicated rows: 0


In [68]:
print(
    "Titles with markup:",
    kaust_crossref_clean["title"]
    .fillna("")
    .str.contains(r"<[^>]+>", regex=True)
    .sum()
)

print(
    "Abstracts with markup:",
    kaust_crossref_clean["abstract"]
    .fillna("")
    .str.contains(r"<[^>]+>", regex=True)
    .sum()
)

Titles with markup: 0
Abstracts with markup: 0


In [69]:
print("Final rows:", len(kaust_crossref_clean))
print("Final columns:", len(kaust_crossref_clean.columns))

print("Missing research IDs:", kaust_crossref_clean["research_id"].isna().sum())
print("Duplicate research IDs:", kaust_crossref_clean["research_id"].duplicated().sum())

kaust_crossref_clean.isna().sum()

Final rows: 124
Final columns: 13
Missing research IDs: 0
Duplicate research IDs: 0


research_id           0
university            0
title                 0
authors               0
publication_year      0
publication_date      1
abstract             16
research_field      124
tech_category       124
journal              46
doi                   0
url                   0
source                0
dtype: int64

In [70]:
output_file = interim_dir / "KAUST_2024_2025_cleaned.csv"

kaust_crossref_clean.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: ..\data\interim\KAUST_2024_2025_cleaned.csv


In [71]:
saved_crossref = pd.read_csv(output_file)

print("Saved rows:", len(saved_crossref))
print("Saved columns:", len(saved_crossref.columns))

Saved rows: 124
Saved columns: 13


## KFUPM - Data Profiling and Cleaning

KFUPM Computer Engineering records from 2023–2026 are loaded from the raw JSON files, combined, profiled, and cleaned.

In [72]:
kfupm_files = sorted(
    raw_dir.glob("KFUPM_C1_*_raw.json")
)

print("Files found:", len(kfupm_files))

Files found: 4


In [73]:
kfupm_records = []

for file in kfupm_files:
    with open(file, "r", encoding="utf-8") as f:
        records = json.load(f)

    kfupm_records.extend(records)

print("Total KFUPM records:", len(kfupm_records))

Total KFUPM records: 48


In [74]:
kfupm = pd.json_normalize(kfupm_records)

print("Rows:", kfupm.shape[0])
print("Columns:", kfupm.shape[1])

Rows: 48
Columns: 33


In [75]:
kfupm.columns.tolist()

['copyright',
 'status_changed',
 'subjects',
 'dir',
 'advisor_approved',
 'members',
 'documents',
 'rev_number',
 'eprint_status',
 'full_text_status',
 'english_abstract',
 'datestamp',
 'thesis_type',
 'institution',
 'date',
 'pages',
 'uri',
 'userid',
 'ispublished',
 'contact_email',
 'co_advisor',
 'divisions',
 'eprintid',
 'lastmod',
 'creators',
 'arabic_abstract',
 'metadata_visibility',
 'advisor',
 'title',
 'type',
 'projects',
 'funders',
 'related_url']

In [76]:
kfupm.head()

,copyright,status_changed,subjects,dir,advisor_approved,members,documents,rev_number,eprint_status,full_text_status,...,lastmod,creators,arabic_abstract,metadata_visibility,advisor,title,type,projects,funders,related_url
0,TRUE,2023-02-16 05:47:51,"[F10, F18, F21]",disk0/00/14/23/40,TRUE,"[{'name': {'lineage': None, 'given': 'Aiman', ...","[{'formatdesc': 'PhD Dissertation', 'placement...",17,archive,public,...,2026-06-30 09:16:16,"[{'name': {'family': 'Shawahna', 'lineage': No...",أثبتت الشبكات العصبية للتعلم العميق فعاليتها ف...,show,"[{'id': 'sadiq@kfupm.edu.sa', 'name': {'family...",On the Optimal Deployment of Deep Learning Neu...,thesis,NaN,NaN,NaN
1,TRUE,2023-01-10 06:19:41,[F10],disk0/00/14/23/14,TRUE,"[{'id': 'alisuwaiyan@kfupm.edu.sa', 'name': {'...","[{'language': 'en', 'docid': 117603, 'format':...",15,archive,public,...,2026-06-30 09:16:11,"[{'name': {'family': 'Elalfy', 'given': 'Yasse...",تُعد الملاحة الذاتية أحد مجالات البحث النشطة ا...,show,"[{'id': 'ubaroudi@kfupm.edu.sa', 'name': {'lin...",Monocular Depth Estimation Using Deep Learning...,thesis,NaN,NaN,NaN
2,TRUE,2024-12-26 06:32:02,"[F10, F18]",disk0/00/14/31/71,TRUE,"[{'id': 'selferik@kfupm.edu.sa', 'name': {'lin...",[{'files': [{'hash': '441406a4736baf5b36fbb5ae...,13,archive,public,...,2026-07-12 10:15:03,"[{'name': {'family': 'Hasan', 'given': 'Shihab...",شهدت التجارة الإلكترونية توسعًا سريعًا، مما زا...,show,"[{'name': {'family': 'Sheltami', 'lineage': No...",Optimizing Last-Mile Delivery with Hybrid Truc...,thesis,NaN,NaN,NaN
3,TRUE,2024-12-25 10:08:42,"[F10, F18]",disk0/00/14/31/35,TRUE,"[{'id': 'tarek@kfupm.edu.sa', 'name': {'honour...","[{'mime_type': 'application/pdf', 'uri': 'http...",11,archive,public,...,2026-06-30 09:17:55,"[{'name': {'lineage': None, 'honourific': None...",تتطلب التوسعات السريعة في الشبكات اللاسلكية تخ...,show,"[{'id': 'barnawi@kfupm.edu.sa', 'name': {'give...",Dynamic Spectrum Sharing in Heterogenous Wirel...,thesis,NaN,NaN,NaN
4,TRUE,2024-12-24 11:21:48,"[F10, F18, F21]",disk0/00/14/31/36,TRUE,"[{'name': {'lineage': None, 'given': 'Ahmad', ...","[{'files': [{'mtime': '2024-12-19 11:07:08', '...",13,archive,public,...,2026-06-30 09:17:56,"[{'name': {'honourific': None, 'given': 'Abrar...",تقدم شبكات الجيل الخامس (5G) وعودًا بدمج العدي...,show,"[{'id': 'elrabaa@kfupm.edu.sa', 'name': {'line...",Dos Attack Against 5g Core Network in 5g- Base...,thesis,NaN,NaN,NaN


In [77]:
for column in kfupm.columns:
    sample = kfupm[column].dropna()

    if not sample.empty:
        value = sample.iloc[0]

        if isinstance(value, (list, dict)):
            print(column, "->", type(value).__name__)

subjects -> list
members -> list
documents -> list
co_advisor -> list
divisions -> list
creators -> list
advisor -> list
projects -> list
funders -> list
related_url -> list


In [78]:
kfupm_profile = pd.DataFrame({
    "column": kfupm.columns,
    "dtype": kfupm.dtypes.astype(str).values,
    "null_count": kfupm.isnull().sum().values,
    "null_percent": (
        kfupm.isnull().mean().values * 100
    ).round(2),
    "unique_count": [
        safe_unique_count(kfupm[col])
        for col in kfupm.columns
    ]
})

kfupm_profile


,column,dtype,null_count,null_percent,unique_count
0,copyright,str,0,0.00,1
1,status_changed,str,0,0.00,48
2,subjects,object,0,0.00,20
3,dir,str,0,0.00,48
4,advisor_approved,str,0,0.00,1
5,members,object,0,0.00,40
6,documents,object,0,0.00,48
7,rev_number,int64,0,0.00,14
8,eprint_status,str,0,0.00,1
9,full_text_status,str,0,0.00,2


In [79]:
important_columns = [
    "title",
    "creators",
    "date",
    "uri",
    "english_abstract",
    "thesis_type",
    "subjects",
    "divisions",
    "institution",
    "type"
]

for column in important_columns:
    if column in kfupm.columns:
        print(
            column,
            "- missing:",
            kfupm[column].isna().sum()
        )

title - missing: 0
creators - missing: 0
date - missing: 0
uri - missing: 0
english_abstract - missing: 0
thesis_type - missing: 0
subjects - missing: 0
divisions - missing: 0
institution - missing: 0
type - missing: 0


In [80]:
for column in ["creators", "subjects", "divisions"]:
    print("\n", column)
    print(kfupm[column].iloc[0])


 creators
[{'name': {'family': 'Shawahna', 'lineage': None, 'given': 'Ahmad', 'honourific': None}}]

 subjects
['F10', 'F18', 'F21']

 divisions
['C1']


In [81]:
kfupm_clean = kfupm.copy()

In [82]:
def extract_kfupm_authors(creators):
    names = []

    for creator in creators:
        name = creator.get("name", {})

        given = name.get("given", "")
        family = name.get("family", "")

        full_name = f"{given} {family}".strip()

        if full_name:
            names.append(full_name)

    return "; ".join(names)


kfupm_clean["authors"] = kfupm_clean["creators"].apply(
    extract_kfupm_authors
)

In [83]:
kfupm_clean["subject_codes"] = kfupm_clean["subjects"].apply(
    lambda x: "; ".join(x)
    if isinstance(x, list)
    else pd.NA
)

In [84]:
kfupm_clean["publication_year"] = pd.to_numeric(
    kfupm_clean["date"],
    errors="coerce"
).astype("Int64")

kfupm_clean["publication_date"] = pd.NaT

In [85]:
for column in ["title", "english_abstract"]:
    kfupm_clean[column] = (
        kfupm_clean[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [86]:
kfupm_clean["research_id"] = (
    "KFUPM-" + kfupm_clean["eprintid"].astype(str)
)

kfupm_clean["university"] = "KFUPM"

kfupm_clean["abstract"] = (
    kfupm_clean["english_abstract"]
)

kfupm_clean["research_field"] = "Computer Engineering"

kfupm_clean["tech_category"] = pd.NA

kfupm_clean["journal"] = pd.NA

kfupm_clean["doi"] = pd.NA

kfupm_clean["url"] = kfupm_clean["uri"]

kfupm_clean["source"] = "KFUPM EPrints"

In [87]:
kfupm_clean = kfupm_clean[
    [
        "research_id",
        "university",
        "title",
        "authors",
        "publication_year",
        "publication_date",
        "abstract",
        "research_field",
        "tech_category",
        "journal",
        "doi",
        "url",
        "source"
    ]
].copy()

kfupm_clean.head()

,research_id,university,title,authors,publication_year,publication_date,abstract,research_field,tech_category,journal,doi,url,source
0,KFUPM-142340,KFUPM,On the Optimal Deployment of Deep Learning Neu...,Ahmad Shawahna,2023,NaT,Deep learning neural networks (DNNs) have demo...,Computer Engineering,<NA>,<NA>,<NA>,https://eprints.kfupm.edu.sa/id/eprint/142340,KFUPM EPrints
1,KFUPM-142314,KFUPM,Monocular Depth Estimation Using Deep Learning...,Yasser Elalfy,2023,NaT,Autonomous navigation is one of the active res...,Computer Engineering,<NA>,<NA>,<NA>,https://eprints.kfupm.edu.sa/id/eprint/142314,KFUPM EPrints
2,KFUPM-143171,KFUPM,Optimizing Last-Mile Delivery with Hybrid Truc...,Shihab Yaqoub Ali Hasan,2024,NaT,The rapid expansion of e-commerce has heighten...,Computer Engineering,<NA>,<NA>,<NA>,https://eprints.kfupm.edu.sa/id/eprint/143171,KFUPM EPrints
3,KFUPM-143135,KFUPM,Dynamic Spectrum Sharing in Heterogenous Wirel...,Sulaimon Adebayo,2024,NaT,The rapid expansion of wireless networks deman...,Computer Engineering,<NA>,<NA>,<NA>,https://eprints.kfupm.edu.sa/id/eprint/143135,KFUPM EPrints
4,KFUPM-143136,KFUPM,Dos Attack Against 5g Core Network in 5g- Base...,Abrar Alqahtani,2024,NaT,5G networks promise to integrate several new t...,Computer Engineering,<NA>,<NA>,<NA>,https://eprints.kfupm.edu.sa/id/eprint/143136,KFUPM EPrints


In [88]:
print(
    "Duplicate research IDs:",
    kfupm_clean.duplicated(
        subset=["research_id"]
    ).sum()
)

print(
    "Duplicate URLs:",
    kfupm_clean.duplicated(
        subset=["url"]
    ).sum()
)

print(
    "Duplicate titles:",
    kfupm_clean.duplicated(
        subset=["title"]
    ).sum()
)

print(
    "Fully duplicated rows:",
    kfupm_clean.duplicated().sum()
)

Duplicate research IDs: 0
Duplicate URLs: 0
Duplicate titles: 0
Fully duplicated rows: 0


In [89]:
kfupm_clean = kfupm.copy()

In [90]:
def extract_kfupm_authors(creators):
    if not isinstance(creators, list):
        return pd.NA

    names = []

    for creator in creators:
        name = creator.get("name", {})
        given = name.get("given", "")
        family = name.get("family", "")

        full_name = f"{given} {family}".strip()

        if full_name:
            names.append(full_name)

    return "; ".join(names)

kfupm_clean["authors"] = kfupm_clean["creators"].apply(
    extract_kfupm_authors
)

In [91]:
kfupm_clean[
    ["title", "creators", "authors"]
].head()

,title,creators,authors
0,On the Optimal Deployment of Deep Learning Neu...,"[{'name': {'family': 'Shawahna', 'lineage': No...",Ahmad Shawahna
1,Monocular Depth Estimation Using Deep Learning...,"[{'name': {'family': 'Elalfy', 'given': 'Yasse...",Yasser Elalfy
2,Optimizing Last-Mile Delivery with Hybrid Truc...,"[{'name': {'family': 'Hasan', 'given': 'Shihab...",Shihab Yaqoub Ali Hasan
3,Dynamic Spectrum Sharing in Heterogenous Wirel...,"[{'name': {'lineage': None, 'honourific': None...",Sulaimon Adebayo
4,Dos Attack Against 5g Core Network in 5g- Base...,"[{'name': {'honourific': None, 'given': 'Abrar...",Abrar Alqahtani


In [92]:
kfupm_clean["subject_codes"] = kfupm_clean["subjects"].apply(
    lambda x: "; ".join(x) if isinstance(x, list) else pd.NA
)

kfupm_clean[
    ["title", "subjects", "subject_codes"]
].head()

,title,subjects,subject_codes
0,On the Optimal Deployment of Deep Learning Neu...,"[F10, F18, F21]",F10; F18; F21
1,Monocular Depth Estimation Using Deep Learning...,[F10],F10
2,Optimizing Last-Mile Delivery with Hybrid Truc...,"[F10, F18]",F10; F18
3,Dynamic Spectrum Sharing in Heterogenous Wirel...,"[F10, F18]",F10; F18
4,Dos Attack Against 5g Core Network in 5g- Base...,"[F10, F18, F21]",F10; F18; F21


In [93]:
kfupm_clean["publication_year"] = pd.to_numeric(
    kfupm_clean["date"],
    errors="coerce"
).astype("Int64")

kfupm_clean["publication_date"] = pd.NaT

print(
    kfupm_clean["publication_year"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "Missing publication years:",
    kfupm_clean["publication_year"].isna().sum()
)

publication_year
2023     2
2024    16
2025    13
2026    17
Name: count, dtype: Int64
Missing publication years: 0


In [94]:
for column in ["title", "english_abstract"]:
    kfupm_clean[column] = (
        kfupm_clean[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

kfupm_clean[
    ["title", "english_abstract"]
].head()

,title,english_abstract
0,On the Optimal Deployment of Deep Learning Neu...,Deep learning neural networks (DNNs) have demo...
1,Monocular Depth Estimation Using Deep Learning...,Autonomous navigation is one of the active res...
2,Optimizing Last-Mile Delivery with Hybrid Truc...,The rapid expansion of e-commerce has heighten...
3,Dynamic Spectrum Sharing in Heterogenous Wirel...,The rapid expansion of wireless networks deman...
4,Dos Attack Against 5g Core Network in 5g- Base...,5G networks promise to integrate several new t...


In [95]:
print("Missing eprintid:", kfupm_clean["eprintid"].isna().sum())
print("Duplicate eprintid:", kfupm_clean["eprintid"].duplicated().sum())

print("Missing URI:", kfupm_clean["uri"].isna().sum())
print("Duplicate URI:", kfupm_clean["uri"].duplicated().sum())

Missing eprintid: 0
Duplicate eprintid: 0
Missing URI: 0
Duplicate URI: 0


In [96]:
kfupm_clean["research_id"] = (
    "KFUPM-" + kfupm_clean["eprintid"].astype("string")
)

kfupm_clean["university"] = "KFUPM"
kfupm_clean["abstract"] = kfupm_clean["english_abstract"]
kfupm_clean["research_field"] = "Computer Engineering"
kfupm_clean["tech_category"] = pd.NA
kfupm_clean["journal"] = pd.NA
kfupm_clean["doi"] = pd.NA
kfupm_clean["url"] = kfupm_clean["uri"]
kfupm_clean["source"] = "KFUPM EPrints"

In [97]:
print("Missing research IDs:", kfupm_clean["research_id"].isna().sum())
print("Duplicate research IDs:", kfupm_clean["research_id"].duplicated().sum())

print("Missing titles:", kfupm_clean["title"].isna().sum())
print("Missing authors:", kfupm_clean["authors"].isna().sum())
print("Missing abstracts:", kfupm_clean["abstract"].isna().sum())

Missing research IDs: 0
Duplicate research IDs: 0
Missing titles: 0
Missing authors: 0
Missing abstracts: 0


In [98]:
print(
    "Duplicate titles:",
    kfupm_clean["title"].duplicated().sum()
)

duplicate_titles = kfupm_clean[
    kfupm_clean["title"].duplicated(keep=False)
].sort_values("title")

duplicate_titles[
    [
        "eprintid",
        "title",
        "authors",
        "publication_year",
        "uri"
    ]
]

Duplicate titles: 0


,eprintid,title,authors,publication_year,uri


In [99]:
kfupm_clean = kfupm_clean[
    [
        "research_id",
        "university",
        "title",
        "authors",
        "publication_year",
        "publication_date",
        "abstract",
        "research_field",
        "tech_category",
        "journal",
        "doi",
        "url",
        "source"
    ]
].copy()

In [100]:
print("Final rows:", len(kfupm_clean))
print("Final columns:", len(kfupm_clean.columns))

print("Missing research IDs:", kfupm_clean["research_id"].isna().sum())
print("Duplicate research IDs:", kfupm_clean["research_id"].duplicated().sum())

kfupm_clean.isna().sum()

Final rows: 48
Final columns: 13
Missing research IDs: 0
Duplicate research IDs: 0


research_id          0
university           0
title                0
authors              0
publication_year     0
publication_date    48
abstract             0
research_field       0
tech_category       48
journal             48
doi                 48
url                  0
source               0
dtype: int64

In [101]:
output_file = interim_dir / "KFUPM_cleaned.csv"

kfupm_clean.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: ..\data\interim\KFUPM_cleaned.csv


In [102]:
saved_kfupm = pd.read_csv(output_file)

print("Saved rows:", len(saved_kfupm))
print("Saved columns:", len(saved_kfupm.columns))

Saved rows: 48
Saved columns: 13
